# Laboratory Specimen Journey Dashboard

This dashboard visualizes the average timeline of events in the hospital laboratory specimen journey, allowing comparison across test types, departments, and time periods.

**Use the filters below to explore:**
- **Test Code** — filter by a specific lab test (e.g. CBC)
- **Dept** — filter by the fulfilling laboratory department
- **Compare By** — toggle between weekday vs. weekend or day shift vs. night shift comparisons

In [ ]:
%%capture
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from plotly.subplots import make_subplots

# --- load data ---
df = pd.read_csv('2025_specimen_time_series_events_no_phi.tsv', sep='\t')
df['event_dt'] = pd.to_datetime(df['event_dt'])
df['test_id'] = df['accession_id'].astype(str) + '_' + df['test_code']
order_df = df[df['event_source'] == 'order'].copy()

# --- build timeline ---
timeline = order_df.pivot_table(
    index='test_id',
    columns='event_type',
    values='event_dt',
    aggfunc='min'
).reset_index()

timeline.columns.name = None
timeline = timeline.rename(columns={
    'collected_dt': 'test_collected_dt',
    'receipt_dt': 'test_receipt_dt',
    'resulted_dt': 'test_min_resulted_dt',
    'verified_dt': 'test_min_verified_dt',
    'cancelled_dt': 'test_cancelled_dt',
    'cancellation_dt': 'test_cancelled_dt'
})

# --- compute durations ---
event_cols = [
    'test_collected_dt', 'test_receipt_dt',
    'test_min_resulted_dt', 'test_min_verified_dt',
    'test_cancelled_dt'
]
existing_event_cols = [col for col in event_cols if col in timeline.columns]
timeline['start'] = timeline['test_ordered_dt']
for col in existing_event_cols:
    timeline[col + '_hrs'] = (timeline[col] - timeline['start']).dt.total_seconds() / 3600

# --- clean bad data ---
timeline = timeline[
    (timeline['test_collected_dt'].isna()) |
    (timeline['test_collected_dt'] >= timeline['test_ordered_dt'])
]

# --- derive extra columns ---
if 'test_code' not in timeline.columns:
    timeline['test_code'] = timeline['test_id'].str.rsplit('_', n=1).str[-1]

timeline['order_day'] = timeline['start'].dt.dayofweek
timeline['group'] = timeline['order_day'].apply(
    lambda x: 'Weekend' if x >= 5 else 'Weekday'
)
timeline['order_hour'] = timeline['start'].dt.hour
timeline['shift'] = timeline['order_hour'].apply(
    lambda h: 'Day (6am–6pm)' if 6 <= h < 18 else 'Night (6pm–6am)'
)

dept_map = order_df[['test_id', 'test_performing_dept']].drop_duplicates('test_id')
timeline = timeline.merge(dept_map, on='test_id', how='left')

timeline['cancelled'] = timeline['test_cancelled_dt'].notna()
timeline['delayed_result'] = timeline['test_min_resulted_dt_hrs'] > 4

# --- build tiered test dropdown ---
test_counts = timeline['test_code'].value_counts()
top_20 = test_counts.head(20).index.tolist()
all_tests = test_counts.index.tolist()

test_options = (
    [('All', 'All'), ('── Top 20 Tests ──', 'Top 20 Tests')]
    + [(f"  {code}", code) for code in top_20]
    + [('── All Tests ──', 'All Tests header')]
    + [(f"  {code}", code) for code in all_tests if code not in top_20]
)

test_dd = widgets.Dropdown(
    options=test_options,
    description='Test Code:',
    value='All',
    layout=widgets.Layout(width='300px')
)

# --- build tiered dept dropdown ---
dept_counts = timeline['test_performing_dept'].value_counts()
top_5_depts = dept_counts.head(5).index.tolist()
all_depts = dept_counts.index.tolist()

dept_options = (
    [('All', 'All'), ('── Top 5 Departments ──', 'Top 5 Depts')]
    + [(f"  {dept}", dept) for dept in top_5_depts]
    + [('── All Departments ──', 'All Depts header')]
    + [(f"  {dept}", dept) for dept in all_depts if dept not in top_5_depts]
)

dept_dd = widgets.Dropdown(
    options=dept_options,
    description='Dept:',
    value='All',
    layout=widgets.Layout(width='300px')
)

ab_options = ['Weekday vs Weekend', 'Day Shift vs Night Shift']
ab_radio = widgets.RadioButtons(
    options=ab_options,
    description='Compare By:',
    value='Weekday vs Weekend'
)

# --- label mappings ---
STAGE_LABELS = {
    'collected_dt': 'Collected',
    'receipt_dt': 'Received at Lab',
    'resulted_dt': 'Result Available',
    'verified_dt': 'Verified',
    'ordered': 'Ordered',
    'collected': 'Collected',
    'received': 'Received at Lab',
    'resulted': 'Result Available',
    'verified': 'Verified',
}

EVENT_LABELS = {
    'cancelled': 'Cancelled',
    'delayed_result': 'Delayed Result (>4hrs)',
}

GROUP_LABELS = {
    'Weekday': 'Weekday',
    'Weekend': 'Weekend',
    'Day (6am–6pm)': 'Day Shift (6am–6pm)',
    'Night (6pm–6am)': 'Night Shift (6pm–6am)',
}

COLOR_MAP = {
    'Weekday': '#4C78A8',
    'Weekend': '#F58518',
    'Day Shift (6am–6pm)': '#54A24B',
    'Night Shift (6pm–6am)': '#B279A2'
}

def fmt_hours(h):
    if pd.isna(h):
        return 'N/A'
    hours = int(h)
    minutes = round((h - hours) * 60)
    if hours == 0:
        return f"{minutes}m"
    elif minutes == 0:
        return f"{hours}h"
    else:
        return f"{hours}h {minutes}m"

### Filters

In [ ]:
def render_dashboard(test_code, dept, compare_by):

    # --- apply filters ---
    data = timeline.copy()

    if test_code == 'All':
        pass
    elif test_code == 'Top 20 Tests':
        data = data[data['test_code'].isin(top_20)]
    elif test_code == 'All Tests header':
        pass
    else:
        data = data[data['test_code'] == test_code]

    if dept == 'All':
        pass
    elif dept == 'Top 5 Depts':
        data = data[data['test_performing_dept'].isin(top_5_depts)]
    elif dept == 'All Depts header':
        pass
    else:
        data = data[data['test_performing_dept'] == dept]

    # --- assign A/B group ---
    data = data.copy()
    if compare_by == 'Weekday vs Weekend':
        data['ab_group'] = data['group']
    else:
        data['ab_group'] = data['shift']

    data['ab_group'] = data['ab_group'].map(GROUP_LABELS).fillna(data['ab_group'])

    # --- split into completed vs all ---
    data_completed = data[data['cancelled'] == False].copy()

    n = len(data)
    n_completed = len(data_completed)

    # --- melt for timeline (completed only, no cancelled column) ---
    duration_cols = [c for c in data_completed.columns if c.endswith('_hrs') and 'cancelled' not in c]
    long_df = data_completed.melt(
        id_vars=['test_id', 'ab_group'],
        value_vars=duration_cols,
        var_name='event_stage',
        value_name='hours'
    )
    long_df['event_stage'] = (
        long_df['event_stage']
        .str.replace('_hrs', '', regex=False)
        .str.replace('test_', '', regex=False)
        .str.replace('min_', '', regex=False)
    )
    long_df['event_stage'] = long_df['event_stage'].map(STAGE_LABELS).fillna(long_df['event_stage'])
    long_df['hours'] = long_df['hours'].round(2)

    ab_df = (
        long_df.groupby(['ab_group', 'event_stage'])['hours']
        .mean().round(2).reset_index()
    )
    ab_df['label'] = ab_df['hours'].apply(fmt_hours)

    stage_order = ['Collected', 'Received at Lab', 'Result Available', 'Verified']

    # --- Chart 1: A/B timeline ---
    fig_timeline = px.line(
        ab_df, x='event_stage', y='hours',
        color='ab_group', markers=True,
        color_discrete_map=COLOR_MAP,
        custom_data=['label'],
        labels={
            'event_stage': 'Specimen Journey Stage',
            'hours': 'Hours from Order',
            'ab_group': 'Group'
        },
        title='Average Specimen Journey Timeline'
    )
    fig_timeline.update_layout(
        hovermode='x unified',
        xaxis={'categoryorder': 'array', 'categoryarray': stage_order}
    )
    fig_timeline.update_traces(hovertemplate='%{customdata[0]}')
    fig_timeline.show()

    # --- Chart 2: Event likelihood ---
    prob_df = data.groupby('ab_group').agg(
        cancelled=('cancelled', 'mean'),
        delayed_result=('delayed_result', 'mean')
    ).reset_index().melt(
        id_vars='ab_group',
        var_name='event_type',
        value_name='probability'
    )
    prob_df['probability'] = prob_df['probability'].round(4)
    prob_df['event_type'] = prob_df['event_type'].map(EVENT_LABELS).fillna(prob_df['event_type'])

    fig_prob = px.bar(
        prob_df, x='event_type', y='probability',
        color='ab_group', barmode='group',
        color_discrete_map=COLOR_MAP,
        text_auto='.1%',
        labels={
            'event_type': 'Event Type',
            'probability': 'Likelihood',
            'ab_group': 'Group'
        },
        title='Likelihood of Key Events'
    )
    fig_prob.update_layout(yaxis_tickformat=',.0%')
    fig_prob.update_traces(hovertemplate='%{y:.1%}')
    fig_prob.show()

    # --- Chart 3: Drop-off between stages ---
    stage_cols = [
        ('Ordered',          'test_ordered_dt'),
        ('Collected',        'test_collected_dt'),
        ('Received at Lab',  'test_receipt_dt'),
        ('Result Available', 'test_min_resulted_dt'),
        ('Verified',         'test_min_verified_dt'),
    ]
    stage_cols = [(label, col) for label, col in stage_cols if col in data_completed.columns]
    groups = sorted(data_completed['ab_group'].dropna().unique())

    dropoff_rows = []
    transitions = [
        (f"{stage_cols[i][0]} → {stage_cols[i+1][0]}", stage_cols[i][1], stage_cols[i+1][1])
        for i in range(len(stage_cols) - 1)
    ]
    for grp in groups:
        grp_data = data_completed[data_completed['ab_group'] == grp]
        for label, from_col, to_col in transitions:
            from_count = grp_data[from_col].notna().sum()
            to_count = grp_data[to_col].notna().sum()
            pct_lost = round((1 - to_count / from_count) * 100, 2) if from_count > 0 else 0
            dropoff_rows.append({'Group': grp, 'Transition': label, 'Percent Lost (%)': pct_lost})

    dropoff_df = pd.DataFrame(dropoff_rows)

    fig_dropoff = px.bar(
        dropoff_df,
        x='Transition', y='Percent Lost (%)',
        color='Group', barmode='group',
        color_discrete_map=COLOR_MAP,
        text_auto='.2f',
        title='Specimen Drop-off Between Stages'
    )
    fig_dropoff.update_layout(
        xaxis_title='Stage Transition',
        yaxis_title='Specimens Lost (%)'
    )
    fig_dropoff.update_traces(hovertemplate='%{y:.2f}%')
    fig_dropoff.show()


widgets.interactive(render_dashboard,
                    test_code=test_dd,
                    dept=dept_dd,
                    compare_by=ab_radio)

interactive(children=(Dropdown(description='Test Code:', layout=Layout(width='300px'), options=(('All', 'All')…